In [ ]:
# Cell 0: Setup — install packages + mount Drive
!pip install -q torch transformers==5.6.2 datasets sentencepiece seqeval pyarrow huggingface_hub
from google.colab import drive
drive.mount('/content/drive')
import torch, transformers
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"transformers: {transformers.__version__}")
print(f"torch: {torch.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, VRAM: 102.0 GB
transformers: 5.6.2
torch: 2.10.0+cu128


In [ ]:
# Cell 1: Clone repo + download data + ensure directories
import os, subprocess
if not os.path.exists("/content/kniv-nlp-models"):
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/rustic-ai/kniv-nlp-models.git"], cwd="/content")
from huggingface_hub import snapshot_download, hf_hub_download
snapshot_download("dragonscale-ai/kniv-corpus-en", repo_type="dataset",
                  allow_patterns="prepared/kniv-deberta-cascade/*", local_dir="/content/kniv-data")
os.chdir("/content/kniv-nlp-models")
os.makedirs("data/prepared", exist_ok=True)
if not os.path.exists("data/prepared/kniv-deberta-cascade"):
    os.symlink("/content/kniv-data/prepared/kniv-deberta-cascade", "data/prepared/kniv-deberta-cascade")

# Download teacher model from HuggingFace
TEACHER_DIR = "/content/drive/MyDrive/kniv-models/teacher"
os.makedirs(TEACHER_DIR, exist_ok=True)
if not os.path.exists(f"{TEACHER_DIR}/model.pt"):
    print("Downloading teacher model...")
    for fname in ["model.pt", "metadata.json", "tokenizer.json", "spm.model",
                  "tokenizer_config.json", "special_tokens_map.json"]:
        hf_hub_download("dragonscale-ai/kniv-deberta-nlp-base-en-large", fname, local_dir=TEACHER_DIR)
print(f"Teacher model: {os.path.getsize(f'{TEACHER_DIR}/model.pt') / 1e9:.2f} GB")
print(f"Working dir: {os.getcwd()}")

Fetching 40 files:   0%|          | 0/40 [00:00<?, ?it/s]

Teacher model: 1.77 GB
Working dir: /content/kniv-nlp-models


In [ ]:
# Cell 2: Imports + label sets + shared model components

import json, random, re, os, shutil, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm import tqdm

device = torch.device("cuda")
DATA = "data/prepared/kniv-deberta-cascade"

# Distillation v2 directory — fresh location to avoid conflicts with old data
DISTILL_DIR = "/content/drive/MyDrive/kniv-models/distillation_v2"
STUDENT_DIR = "/content/drive/MyDrive/kniv-models/students_v2"

# ── Label sets (same as before) ────────────────────────
POS_LABELS = ["ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ","NOUN","NUM","PART","PRON","PROPN","PUNCT","SCONJ","SYM","VERB","X"]
NER_LABELS = ["O","B-PERSON","I-PERSON","B-NORP","I-NORP","B-FAC","I-FAC","B-ORG","I-ORG","B-GPE","I-GPE","B-LOC","I-LOC","B-PRODUCT","I-PRODUCT","B-EVENT","I-EVENT","B-WORK_OF_ART","I-WORK_OF_ART","B-LAW","I-LAW","B-LANGUAGE","I-LANGUAGE","B-DATE","I-DATE","B-TIME","I-TIME","B-PERCENT","I-PERCENT","B-MONEY","I-MONEY","B-QUANTITY","I-QUANTITY","B-ORDINAL","I-ORDINAL","B-CARDINAL","I-CARDINAL"]
SRL_TAGS = ["O","V","B-ARG0","I-ARG0","B-ARG1","I-ARG1","B-ARG2","I-ARG2","B-ARG3","I-ARG3","B-ARG4","I-ARG4","B-ARGM-TMP","I-ARGM-TMP","B-ARGM-LOC","I-ARGM-LOC","B-ARGM-MNR","I-ARGM-MNR","B-ARGM-CAU","I-ARGM-CAU","B-ARGM-PRP","I-ARGM-PRP","B-ARGM-NEG","I-ARGM-NEG","B-ARGM-ADV","I-ARGM-ADV","B-ARGM-DIR","I-ARGM-DIR","B-ARGM-DIS","I-ARGM-DIS","B-ARGM-EXT","I-ARGM-EXT","B-ARGM-MOD","I-ARGM-MOD","B-ARGM-PRD","I-ARGM-PRD","B-ARGM-GOL","I-ARGM-GOL","B-ARGM-COM","I-ARGM-COM","B-ARGM-REC","I-ARGM-REC"]
CLS_LABELS = ["inform","request","question","confirm","reject","offer","social","status"]
DEPREL_LIST = ["root","acl","acl:relcl","advcl","advcl:relcl","advmod","amod","appos","aux","aux:pass","case","cc","cc:preconj","ccomp","compound","compound:prt","conj","cop","csubj","csubj:outer","csubj:pass","dep","det","det:predet","discourse","dislocated","expl","fixed","flat","goeswith","iobj","list","mark","nmod","nmod:desc","nmod:npmod","nmod:poss","nmod:tmod","nsubj","nsubj:outer","nsubj:pass","nummod","obj","obl","obl:agent","obl:npmod","obl:tmod","orphan","parataxis","punct","reparandum","vocative","xcomp"]
N_POS, N_NER, N_SRL, N_DEP, N_CLS = len(POS_LABELS), len(NER_LABELS), len(SRL_TAGS), len(DEPREL_LIST), len(CLS_LABELS)
pos_map = {l: i for i, l in enumerate(POS_LABELS)}
ner_map = {l: i for i, l in enumerate(NER_LABELS)}

# ── Model components (same as before) ────────────────
class ScalarMix(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.weights = nn.Parameter(torch.zeros(n))
        self.scale = nn.Parameter(torch.ones(1))
    def forward(self, layers):
        w = torch.softmax(self.weights, dim=0)
        return self.scale * sum(wi * li for wi, li in zip(w, layers))

class AttentionPool(nn.Module):
    def __init__(self, H):
        super().__init__()
        self.attn = nn.Linear(H, 1)
    def forward(self, hidden, mask):
        scores = self.attn(hidden).squeeze(-1).masked_fill(~mask.bool(), -1e9)
        return (hidden * torch.softmax(scores, -1).unsqueeze(-1)).sum(1)

class Biaffine(nn.Module):
    def __init__(self, in_dim, out_dim=1):
        super().__init__()
        self.weight = nn.Parameter(torch.zeros(out_dim, in_dim+1, in_dim+1))
        nn.init.xavier_uniform_(self.weight)
    def forward(self, h_dep, h_head):
        B, S, D = h_dep.size()
        h_dep = torch.cat([h_dep, torch.ones(B,S,1,device=h_dep.device)], -1)
        h_head = torch.cat([h_head, torch.ones(B,S,1,device=h_head.device)], -1)
        scores = torch.einsum("bxi,oij,byj->boxy", h_dep, self.weight, h_head)
        return scores.squeeze(1).contiguous() if scores.size(1)==1 else scores.permute(0,2,3,1).contiguous()

class BiaffineDEPHead(nn.Module):
    def __init__(self, H, arc_dim, label_dim, num_labels, dropout=0.1):
        super().__init__()
        self.arc_dep = nn.Sequential(nn.Linear(H, arc_dim), nn.LayerNorm(arc_dim), nn.GELU(), nn.Dropout(dropout))
        self.arc_head = nn.Sequential(nn.Linear(H, arc_dim), nn.LayerNorm(arc_dim), nn.GELU(), nn.Dropout(dropout))
        self.label_dep = nn.Sequential(nn.Linear(H, label_dim), nn.LayerNorm(label_dim), nn.GELU(), nn.Dropout(dropout))
        self.label_head = nn.Sequential(nn.Linear(H, label_dim), nn.LayerNorm(label_dim), nn.GELU(), nn.Dropout(dropout))
        self.biaffine_arc = Biaffine(arc_dim, 1)
        self.biaffine_label = Biaffine(label_dim, num_labels)
    def forward(self, hidden):
        return (self.biaffine_arc(self.arc_dep(hidden), self.arc_head(hidden)),
                self.biaffine_label(self.label_dep(hidden), self.label_head(hidden)))

print(f"Labels: POS={N_POS}, NER={N_NER}, SRL={N_SRL}, DEP={N_DEP}, CLS={N_CLS}")
print(f"Distillation dir (v2): {DISTILL_DIR}")
print(f"Student dir (v2): {STUDENT_DIR}")
print("Setup complete.")

Labels: POS=17, NER=37, SRL=42, DEP=53, CLS=8
Distillation dir (v2): /content/drive/MyDrive/kniv-models/distillation_v2
Student dir (v2): /content/drive/MyDrive/kniv-models/students_v2
Setup complete.


In [ ]:
# Cell 3: Load teacher model — full with all heads + SRL classifier

print("=" * 60)
print("LOADING TEACHER MODEL")
print("=" * 60)

teacher_tok = AutoTokenizer.from_pretrained(TEACHER_DIR)
teacher_enc = AutoModel.from_pretrained("microsoft/deberta-v3-large").float()
H_t = teacher_enc.config.hidden_size
NL_t = teacher_enc.config.num_hidden_layers + 1
print(f"Teacher: {H_t}d, {NL_t-1} layers")

state = torch.load(f"{TEACHER_DIR}/model.pt", map_location="cpu", weights_only=True)
teacher_enc.load_state_dict(state["deberta"])

pred_emb = nn.Embedding(2, H_t); pred_emb.load_state_dict(state["pred_embedding"])
t_pos_sm = ScalarMix(NL_t); t_pos_sm.load_state_dict(state["pos_scalar_mix"])
t_pos_head = nn.Linear(H_t, N_POS); t_pos_head.load_state_dict(state["pos_head"])
t_ner_sm = ScalarMix(NL_t); t_ner_sm.load_state_dict(state["ner_scalar_mix"])
t_ner_lstm = nn.LSTM(H_t, 256, bidirectional=True, batch_first=True); t_ner_lstm.load_state_dict(state["ner_lstm"])
t_ner_proj = nn.Linear(512, H_t); t_ner_proj.load_state_dict(state["ner_proj"])
t_ner_head = nn.Sequential(nn.LayerNorm(H_t+N_POS), nn.Linear(H_t+N_POS, H_t), nn.GELU(), nn.Dropout(0.1), nn.Linear(H_t, N_NER))
t_ner_head.load_state_dict(state["ner_head"])
t_srl_cls = nn.Sequential(nn.Dropout(0.1), nn.Linear(H_t, H_t), nn.GELU(), nn.Dropout(0.1), nn.Linear(H_t, N_SRL))
t_srl_cls.load_state_dict(state["classifier"])
t_dep_sm = ScalarMix(NL_t); t_dep_sm.load_state_dict(state["dep_scalar_mix"])
t_dep_proj = nn.Sequential(nn.LayerNorm(H_t+N_POS+N_NER), nn.Linear(H_t+N_POS+N_NER, H_t), nn.GELU())
t_dep_proj.load_state_dict(state["dep_proj"])
t_dep_biaff = BiaffineDEPHead(H_t, 512, 128, N_DEP); t_dep_biaff.load_state_dict(state["dep_biaffine"])
t_cls_sm = ScalarMix(NL_t); t_cls_sm.load_state_dict(state["cls_scalar_mix"])
t_cls_pool = AttentionPool(H_t); t_cls_pool.load_state_dict(state["cls_pool"])
t_cls_head = nn.Sequential(nn.LayerNorm(H_t), nn.Linear(H_t, H_t//2), nn.GELU(), nn.Dropout(0.1), nn.Linear(H_t//2, N_CLS))
t_cls_head.load_state_dict(state["cls_head"])

t_modules = [teacher_enc, pred_emb, t_pos_sm, t_pos_head, t_ner_sm, t_ner_lstm, t_ner_proj, t_ner_head,
             t_srl_cls, t_dep_sm, t_dep_proj, t_dep_biaff, t_cls_sm, t_cls_pool, t_cls_head]
for m in t_modules: m.float().to(device).eval()
del state
torch.cuda.empty_cache()

# Sanity: test forward pass
test_words = ["The", "cat", "sat", "on", "the", "mat", "."]
with torch.no_grad():
    enc = teacher_tok(test_words, is_split_into_words=True, return_tensors="pt", padding=True, truncation=True, max_length=128)
    enc = {k: v.to(device) for k, v in enc.items()}
    emb = teacher_enc.embeddings(enc["input_ids"])
    out = teacher_enc.encoder(emb, enc["attention_mask"], output_hidden_states=True)
    print(f"\n  Forward sanity check:")
    print(f"    Input shape: {enc['input_ids'].shape}")
    print(f"    Hidden states: {len(out.hidden_states)} layers, last shape: {out.hidden_states[-1].shape}")
    print(f"    Output dtype: {out.hidden_states[-1].dtype}")

print(f"\nTeacher loaded: {sum(p.numel() for m in t_modules for p in m.parameters())/1e6:.0f}M params")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

LOADING TEACHER MODEL


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Teacher: 1024d, 24 layers


model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]


  Forward sanity check:
    Input shape: torch.Size([1, 9])
    Hidden states: 25 layers, last shape: torch.Size([1, 9, 1024])
    Output dtype: torch.float32

Teacher loaded: 443M params
GPU memory used: 4.26 GB


In [ ]:
# Cell 4: Load corpus

print("Loading corpus...")
sentences = []
corpus_dir = "/content/kniv-nlp-models/corpus/output/annotated"
if os.path.isdir(corpus_dir):
    for domain_dir in sorted(os.scandir(corpus_dir), key=lambda d: d.name):
        if not domain_dir.is_dir(): continue
        jsonl = os.path.join(domain_dir.path, "annotated.jsonl")
        if not os.path.exists(jsonl): continue
        count = 0
        with open(jsonl) as f:
            for line in f:
                if len(sentences) >= 500000: break
                obj = json.loads(line)
                words = [t["form"] for t in obj.get("tokens", [])]
                if 3 <= len(words) <= 80:
                    sentences.append(words); count += 1
        print(f"  {domain_dir.name}: {count:,}")
        if len(sentences) >= 500000: break

# Add prepared data (UD, NER train) for additional coverage
for fname in ["ud_train.json", "ner_train.json"]:
    path = f"{DATA}/{fname}"
    if os.path.exists(path) and len(sentences) < 500000:
        with open(path) as f: data = json.load(f)
        for ex in data:
            if len(sentences) >= 500000: break
            words = ex.get("words", [])
            if len(words) >= 3: sentences.append(words)
        print(f"  {fname}: → {len(sentences):,} total")

random.seed(42); random.shuffle(sentences)
print(f"\nTotal: {len(sentences):,} sentences")
print(f"Avg length: {np.mean([len(s) for s in sentences]):.1f} words")

Loading corpus...
  ud_train.json: → 11,400 total
  ner_train.json: → 206,520 total

Total: 206,520 sentences
Avg length: 23.5 words


In [ ]:
# Cell 5: Teacher inference functions — generate enriched distillation data
# Per-sentence: existing artifacts + last hidden + mid hidden states
# Per-verb: SRL hard + soft logits + predicate-aware hidden state

MAX_LENGTH = 128

# Layers we keep for hidden state distillation
# Teacher has 25 outputs (embedding + 24 transformer layers)
# Storing layers 12, 18, 24 covers most reasonable student configurations
TEACHER_LAYERS_TO_STORE = [12, 18, 24]

@torch.no_grad()
def teacher_predict_sentence_batch(words_batch):
    """
    Run teacher's main encoder pass (no predicate) for a batch of sentences.
    Returns sentence-level artifacts: logits, hard labels, hidden states.
    """
    encs = teacher_tok(words_batch, is_split_into_words=True, return_tensors="pt",
                       padding=True, truncation=True, max_length=MAX_LENGTH)
    ids = encs["input_ids"].to(device)
    mask = encs["attention_mask"].to(device)
    B, S = ids.size()

    # Forward without predicate embedding (default for POS/NER/DEP/CLS)
    emb = teacher_enc.embeddings(ids)
    enc_out = teacher_enc.encoder(emb, mask, output_hidden_states=True)
    layers = list(enc_out.hidden_states)  # 25 layers (embedding + 24 transformer)

    # All-head logits using sentence-level (no predicate) representations
    pos_logits = t_pos_head(t_pos_sm(layers))
    pos_probs = torch.softmax(pos_logits, -1)
    ner_h = t_ner_sm(layers); lo, _ = t_ner_lstm(ner_h)
    adapted = t_ner_proj(lo) + ner_h
    ner_logits = t_ner_head(torch.cat([adapted, pos_probs], -1))
    ner_probs = torch.softmax(ner_logits, -1)
    dep_h = t_dep_sm(layers)
    dep_proj = t_dep_proj(torch.cat([dep_h, pos_probs, ner_probs], -1))
    arc_scores, label_scores = t_dep_biaff(dep_proj)
    cls_h = t_cls_sm(layers)
    cls_logits = t_cls_head(t_cls_pool(cls_h, mask))

    # Move to CPU for storage
    pos_logits = pos_logits.cpu(); ner_logits = ner_logits.cpu()
    arc_scores = arc_scores.cpu(); label_scores = label_scores.cpu()
    cls_logits = cls_logits.cpu()

    # Stash teacher hidden states for the mapped layers
    teacher_hidden = {l: layers[l].cpu() for l in TEACHER_LAYERS_TO_STORE}

    results = []
    for b in range(B):
        words = words_batch[b]
        n = len(words)
        word_ids = encs.word_ids(b)
        # word_id → first sub-token index
        w2t = {}; prev = None
        for k, wid in enumerate(word_ids):
            if wid is not None and wid != prev: w2t[wid] = k
            prev = wid

        # Word-level POS/NER (project sub-token logits onto words)
        w_pos = np.zeros((n, N_POS), dtype=np.float16)
        w_ner = np.zeros((n, N_NER), dtype=np.float16)
        for wid, tidx in w2t.items():
            if wid < n:
                w_pos[wid] = pos_logits[b, tidx].half().numpy()
                w_ner[wid] = ner_logits[b, tidx].half().numpy()

        # Word-level DEP arc + top-5 labels
        w_arc = np.zeros((n, n), dtype=np.float16)
        for wi in range(n):
            if wi in w2t:
                for wj in range(n):
                    if wj in w2t:
                        w_arc[wi, wj] = arc_scores[b, w2t[wi], w2t[wj]].half().item()
        dep_top5 = []
        arc_preds = w_arc.argmax(axis=-1)
        for wi in range(n):
            hi = int(arc_preds[wi])
            if wi in w2t and hi in w2t:
                scores = label_scores[b, w2t[wi], w2t[hi]].half().numpy()
                t5 = scores.argsort()[-5:][::-1]
                dep_top5.append([(int(idx), float(scores[idx])) for idx in t5])
            else:
                dep_top5.append([(0, 1.0)])

        # Hard labels (argmax of teacher predictions)
        pos_hard = [POS_LABELS[w_pos[i].argmax()] for i in range(n)]
        ner_hard = [NER_LABELS[w_ner[i].argmax()] for i in range(n)]

        # Word-level hidden states for each tracked teacher layer
        # Reduce sub-tokens to first-sub-token representation per word
        word_hidden = {}
        for layer_idx in TEACHER_LAYERS_TO_STORE:
            wh = np.zeros((n, H_t), dtype=np.float16)
            for wid, tidx in w2t.items():
                if wid < n:
                    wh[wid] = teacher_hidden[layer_idx][b, tidx].half().numpy()
            word_hidden[layer_idx] = wh

        results.append({
            "words": words, "n_words": n,
            # Existing artifacts
            "pos_logits": w_pos.tobytes(), "ner_logits": w_ner.tobytes(),
            "dep_arc_scores": w_arc.tobytes(),
            "dep_label_top5": json.dumps(dep_top5),
            "cls_logits": cls_logits[b].half().numpy().tobytes(),
            "pos_hard": pos_hard, "ner_hard": ner_hard,
            # NEW: word-level teacher hidden states
            **{f"hidden_l{l}": word_hidden[l].tobytes() for l in TEACHER_LAYERS_TO_STORE},
        })
    return results, encs, mask, ids


@torch.no_grad()
def teacher_predict_per_verb(words, ids, mask, w2t, verb_indices):
    """
    Run teacher per-verb to get SRL predictions and predicate-aware hidden states.
    Args:
        words: list of words (n words)
        ids: input_ids tensor [1, S] for this single sentence
        mask: attention_mask [1, S]
        w2t: word→sub-token-index dict
        verb_indices: list of word indices that are verbs
    Returns:
        list of {verb_idx, srl_hard (list of n tags), srl_logits (np.float16 [n, N_SRL]),
                 pred_hidden_l24 (np.float16 [n, H_t])}
    """
    n = len(words)
    S = ids.size(1)
    if not verb_indices:
        return []

    per_verb_data = []
    # Run each verb as a separate forward pass (could batch but keeping simple for now)
    for vi in verb_indices:
        if vi not in w2t:
            continue
        # Inject predicate embedding at verb's first sub-token
        indicator = torch.zeros(1, S, dtype=torch.long, device=device)
        indicator[0, w2t[vi]] = 1
        emb = teacher_enc.embeddings(ids) + pred_emb(indicator)
        srl_out = teacher_enc.encoder(emb, mask, output_hidden_states=True)
        last_hidden = srl_out.last_hidden_state  # [1, S, H_t]
        srl_logits = t_srl_cls(last_hidden)  # [1, S, N_SRL]

        last_hidden_cpu = last_hidden.cpu()
        srl_logits_cpu = srl_logits.cpu()

        # Word-level reduction
        w_srl = np.zeros((n, N_SRL), dtype=np.float16)
        w_pred_hidden = np.zeros((n, H_t), dtype=np.float16)
        for wid, tidx in w2t.items():
            if wid < n:
                w_srl[wid] = srl_logits_cpu[0, tidx].half().numpy()
                w_pred_hidden[wid] = last_hidden_cpu[0, tidx].half().numpy()
        srl_hard = [SRL_TAGS[w_srl[i].argmax()] for i in range(n)]

        per_verb_data.append({
            "verb_idx": vi,
            "srl_hard": srl_hard,
            "srl_logits": w_srl.tobytes(),
            "pred_hidden_l24": w_pred_hidden.tobytes(),
        })
    return per_verb_data


def teacher_process_batch(words_batch):
    """Full pipeline: sentence-level + per-verb processing for a batch."""
    sent_results, encs, mask, ids = teacher_predict_sentence_batch(words_batch)
    final_results = []
    for b, sent_record in enumerate(sent_results):
        n = sent_record["n_words"]
        words = sent_record["words"]
        word_ids = encs.word_ids(b)
        w2t = {}; prev = None
        for k, wid in enumerate(word_ids):
            if wid is not None and wid != prev and wid < n: w2t[wid] = k
            prev = wid
        # Find all verbs
        verb_indices = [i for i, p in enumerate(sent_record["pos_hard"]) if p == "VERB"]
        # Per-verb teacher pass
        per_verb = teacher_predict_per_verb(words, ids[b:b+1], mask[b:b+1], w2t, verb_indices)
        sent_record["per_verb"] = json.dumps([
            {"verb_idx": pv["verb_idx"], "srl_hard": pv["srl_hard"]}
            for pv in per_verb
        ])
        # Store byte arrays separately as columns
        sent_record["per_verb_count"] = len(per_verb)
        sent_record["per_verb_srl_logits"] = b"".join([pv["srl_logits"] for pv in per_verb])
        sent_record["per_verb_pred_hidden"] = b"".join([pv["pred_hidden_l24"] for pv in per_verb])
        sent_record["per_verb_indices"] = json.dumps([pv["verb_idx"] for pv in per_verb])
        final_results.append(sent_record)
    return final_results


print("Generation functions defined.")
print(f"  Tracking teacher hidden states from layers: {TEACHER_LAYERS_TO_STORE}")
print(f"  Per-verb processing: yes (every VERB token gets a teacher SRL pass)")

Generation functions defined.
  Tracking teacher hidden states from layers: [12, 18, 24]
  Per-verb processing: yes (every VERB token gets a teacher SRL pass)


In [ ]:
# Cell 6: SMALL SAMPLE TEST — verify pipeline on 100 sentences before scaling

print("=" * 60)
print("SMALL SAMPLE TEST — 100 sentences")
print("=" * 60)

TEST_N = 100
test_sentences = sentences[:TEST_N]
TEST_DIR = f"{DISTILL_DIR}/test_sample"
os.makedirs(TEST_DIR, exist_ok=True)

# RTX PRO 6000 has 102 GB VRAM — use larger batch
BATCH_SIZE = 32

# Run generation
start_time = time.time()
all_results = []
for batch_start in tqdm(range(0, TEST_N, BATCH_SIZE), desc="Generating"):
    batch = test_sentences[batch_start:batch_start + BATCH_SIZE]
    results = teacher_process_batch(batch)
    all_results.extend(results)
elapsed = time.time() - start_time

print(f"\nGeneration time: {elapsed:.1f}s for {TEST_N} sentences")
print(f"  Per-sentence avg: {elapsed/TEST_N*1000:.0f}ms")
print(f"  Estimated for 200K sentences: {elapsed/TEST_N*200000/3600:.1f} hours")
print(f"  GPU memory peak: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

# Save and check size
test_path = f"{TEST_DIR}/test_shard.parquet"
pq.write_table(pa.Table.from_pylist(all_results), test_path)
size_bytes = os.path.getsize(test_path)
print(f"\nParquet size: {size_bytes/1e6:.1f} MB for {TEST_N} sentences")
print(f"  Per-sentence avg: {size_bytes/TEST_N/1024:.0f} KB")
print(f"  Estimated for 200K sentences: {size_bytes/TEST_N*200000/1e9:.1f} GB")

# ── Sanity-check first record ──
print(f"\n{'='*60}")
print("RECORD STRUCTURE SANITY CHECK")
print(f"{'='*60}")
first = all_results[0]
n = first["n_words"]
print(f"First sentence: {' '.join(first['words'])}")
print(f"  n_words: {n}")
print(f"  pos_hard: {first['pos_hard']}")
print(f"  ner_hard: {first['ner_hard']}")
print(f"  Verbs found: {first['per_verb_count']}, indices: {first['per_verb_indices']}")
per_verb_data = json.loads(first["per_verb"])
for pv in per_verb_data:
    vi = pv["verb_idx"]
    verb_word = first["words"][vi] if vi < len(first["words"]) else "?"
    print(f"    Verb '{verb_word}' (idx {vi}): SRL = {pv['srl_hard']}")

# ── Verify byte arrays decode cleanly ──
print(f"\nByte-array integrity check:")
print(f"  pos_logits: {len(first['pos_logits'])} bytes (expect {n*N_POS*2})")
print(f"  hidden_l24: {len(first['hidden_l24'])} bytes (expect {n*H_t*2})")
print(f"  per_verb_pred_hidden: {len(first['per_verb_pred_hidden'])} bytes "
      f"(expect {first['per_verb_count']*n*H_t*2})")
print(f"  per_verb_srl_logits: {len(first['per_verb_srl_logits'])} bytes "
      f"(expect {first['per_verb_count']*n*N_SRL*2})")

# Try decoding hidden state
hidden_l24 = np.frombuffer(first["hidden_l24"], dtype=np.float16).reshape(n, H_t)
print(f"\nDecoded hidden_l24 shape: {hidden_l24.shape}, dtype: {hidden_l24.dtype}")
print(f"  Stats: mean={hidden_l24.astype(np.float32).mean():.3f}, std={hidden_l24.astype(np.float32).std():.3f}")

print(f"\n{'='*60}")
print("✓ Pipeline test PASSED if no errors above")
print(f"{'='*60}")

SMALL SAMPLE TEST — 100 sentences


Generating: 100%|██████████| 4/4 [00:03<00:00,  1.15it/s]



Generation time: 3.5s for 100 sentences
  Per-sentence avg: 35ms
  Estimated for 200K sentences: 1.9 hours
  GPU memory peak: 12.01 GB

Parquet size: 30.7 MB for 100 sentences
  Per-sentence avg: 299 KB
  Estimated for 200K sentences: 61.3 GB

RECORD STRUCTURE SANITY CHECK
First sentence: However , the latest research shows that it was not built until 1888/89 .
  n_words: 14
  pos_hard: ['ADV', 'PUNCT', 'DET', 'ADJ', 'NOUN', 'VERB', 'SCONJ', 'PRON', 'AUX', 'PART', 'VERB', 'ADP', 'NUM', 'PUNCT']
  ner_hard: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-DATE', 'O']
  Verbs found: 2, indices: [5, 10]
    Verb 'shows' (idx 5): SRL = ['B-ARGM-DIS', 'O', 'O', 'B-ARGM-TMP', 'B-ARG0', 'V', 'O', 'O', 'O', 'O', 'B-ARG1', 'O', 'O', 'O']
    Verb 'built' (idx 10): SRL = ['O', 'O', 'O', 'B-ARGM-TMP', 'V', 'O', 'O', 'O', 'O', 'B-ARGM-NEG', 'V', 'O', 'O', 'O']

Byte-array integrity check:
  pos_logits: 476 bytes (expect 476)
  hidden_l24: 28672 bytes (expect 28672)
  per_verb_pred_

In [ ]:
# Cell 7: Throughput optimization test — find optimal batch size
# With 102 GB VRAM and only 12 GB used at batch=32, we can go much higher

print("=" * 60)
print("THROUGHPUT OPTIMIZATION — finding best batch size")
print("=" * 60)

TEST_N = 200  # larger sample for stable timing
test_sentences_2 = sentences[:TEST_N]

results_table = []
for bs in [32, 64, 128, 256]:
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    start = time.time()
    n_processed = 0
    for batch_start in range(0, TEST_N, bs):
        batch = test_sentences_2[batch_start:batch_start + bs]
        _ = teacher_process_batch(batch)
        n_processed += len(batch)
    elapsed = time.time() - start
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    sec_per_sent = elapsed / n_processed
    est_full = sec_per_sent * 200000 / 60
    results_table.append((bs, elapsed, sec_per_sent*1000, peak_mem, est_full))
    print(f"  batch={bs:>4}: {elapsed:.1f}s for {n_processed} sents, "
          f"{sec_per_sent*1000:.0f}ms/sent, peak mem {peak_mem:.1f}GB, "
          f"est. {est_full:.0f} min for 200K")

# Find best batch size
best = min(results_table, key=lambda x: x[2])
print(f"\nBest: batch={best[0]} → {best[2]:.0f}ms/sent, {best[4]:.0f} min total")
OPTIMAL_BATCH = best[0]
print(f"\nOPTIMAL_BATCH set to {OPTIMAL_BATCH}")

THROUGHPUT OPTIMIZATION — finding best batch size
  batch=  32: 6.4s for 200 sents, 32ms/sent, peak mem 5.1GB, est. 107 min for 200K
  batch=  64: 6.5s for 200 sents, 33ms/sent, peak mem 6.0GB, est. 109 min for 200K
  batch= 128: 6.4s for 200 sents, 32ms/sent, peak mem 7.3GB, est. 106 min for 200K
  batch= 256: 6.3s for 200 sents, 32ms/sent, peak mem 9.6GB, est. 105 min for 200K

Best: batch=256 → 32ms/sent, 105 min total

OPTIMAL_BATCH set to 256


In [ ]:
# Cell 8: FULL GENERATION — process all 200K+ sentences and shard to Drive

print("=" * 60)
print("FULL DISTILLATION DATA GENERATION")
print("=" * 60)

os.makedirs(DISTILL_DIR, exist_ok=True)
SHARD_SIZE = 50000
BATCH_SIZE = 64  # sweet spot — anything higher is no faster

# Check for existing data (resume support)
existing_shards = sorted([f for f in os.listdir(DISTILL_DIR) if f.endswith(".parquet") and f.startswith("shard_")])
if existing_shards:
    # Count sentences in existing shards
    n_existing = sum(pq.read_metadata(f"{DISTILL_DIR}/{s}").num_rows for s in existing_shards)
    print(f"Found {len(existing_shards)} existing shards with {n_existing:,} sentences. Resuming from there.")
    sentences_to_process = sentences[n_existing:]
    shard_idx = len(existing_shards)
else:
    sentences_to_process = sentences
    shard_idx = 0
    n_existing = 0

print(f"Sentences to process: {len(sentences_to_process):,}")
print(f"Will write to {DISTILL_DIR}")
print(f"Shard size: {SHARD_SIZE:,}, batch size: {BATCH_SIZE}")
print()

# Generate
shard_rows = []
total = n_existing
start_time = time.time()
last_log = start_time

for batch_start in tqdm(range(0, len(sentences_to_process), BATCH_SIZE), desc="Distilling"):
    batch = sentences_to_process[batch_start:batch_start + BATCH_SIZE]
    results = teacher_process_batch(batch)
    shard_rows.extend(results)
    total += len(results)

    # Save shard when full
    if len(shard_rows) >= SHARD_SIZE:
        path = f"{DISTILL_DIR}/shard_{shard_idx:03d}.parquet"
        pq.write_table(pa.Table.from_pylist(shard_rows), path)
        size_gb = os.path.getsize(path) / 1e9
        elapsed = time.time() - start_time
        rate = (total - n_existing) / elapsed if elapsed > 0 else 0
        eta_min = (len(sentences) - total) / rate / 60 if rate > 0 else 0
        print(f"\n  ✓ Saved {path} ({len(shard_rows):,} rows, {size_gb:.1f} GB) | "
              f"{rate:.0f} sent/s | ETA {eta_min:.0f} min")
        shard_rows = []
        shard_idx += 1

# Save final partial shard
if shard_rows:
    path = f"{DISTILL_DIR}/shard_{shard_idx:03d}.parquet"
    pq.write_table(pa.Table.from_pylist(shard_rows), path)
    size_gb = os.path.getsize(path) / 1e9
    print(f"\n  ✓ Saved {path} ({len(shard_rows):,} rows, {size_gb:.1f} GB)")
    shard_idx += 1

# Write metadata
elapsed_total = time.time() - start_time
total_gb = sum(os.path.getsize(f"{DISTILL_DIR}/{f}") for f in os.listdir(DISTILL_DIR)
               if f.endswith(".parquet") and f.startswith("shard_")) / 1e9

meta = {
    "total_sentences": total,
    "total_shards": shard_idx,
    "total_size_gb": round(total_gb, 2),
    "shard_size": SHARD_SIZE,
    "teacher": "kniv-deberta-nlp-base-en-large",
    "teacher_layers_stored": TEACHER_LAYERS_TO_STORE,
    "max_length": MAX_LENGTH,
    "version": "v2",
    "schema": {
        "existing": ["words", "n_words", "pos_logits", "ner_logits",
                     "dep_arc_scores", "dep_label_top5", "cls_logits",
                     "pos_hard", "ner_hard"],
        "new_v2": [f"hidden_l{l}" for l in TEACHER_LAYERS_TO_STORE] + [
            "per_verb", "per_verb_count", "per_verb_indices",
            "per_verb_srl_logits", "per_verb_pred_hidden"
        ],
    },
}
with open(f"{DISTILL_DIR}/metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"\n{'='*60}")
print(f"FULL GENERATION COMPLETE")
print(f"{'='*60}")
print(f"  Total sentences: {total:,}")
print(f"  Total shards: {shard_idx}")
print(f"  Total size: {total_gb:.1f} GB")
print(f"  Total time: {elapsed_total/60:.0f} min")
print(f"  Saved to: {DISTILL_DIR}")

FULL DISTILLATION DATA GENERATION
Sentences to process: 206,520
Will write to /content/drive/MyDrive/kniv-models/distillation_v2
Shard size: 50,000, batch size: 64



Distilling:   0%|          | 1/3227 [00:02<2:12:37,  2.47s/it]

In [24]:
# Cell 9: Verify generated distillation data on Drive

print("=" * 60)
print("DATA VERIFICATION")
print("=" * 60)

# ── Discover files ──
all_files = sorted(os.listdir(DISTILL_DIR))
shard_files = [f for f in all_files if f.startswith("shard_") and f.endswith(".parquet")]
has_metadata = "metadata.json" in all_files

print(f"\nDirectory: {DISTILL_DIR}")
print(f"  Shards found: {len(shard_files)}")
print(f"  metadata.json: {'yes' if has_metadata else 'NO'}")

# ── Sizes ──
total_bytes = 0
print(f"\nShard breakdown:")
for f in shard_files:
    path = f"{DISTILL_DIR}/{f}"
    size_gb = os.path.getsize(path) / 1e9
    total_bytes += os.path.getsize(path)
    n_rows = pq.read_metadata(path).num_rows
    print(f"  {f}: {size_gb:6.2f} GB  ({n_rows:>6,} rows)")
print(f"  {'─'*60}")
print(f"  TOTAL: {total_bytes/1e9:6.2f} GB")

total_rows = sum(pq.read_metadata(f"{DISTILL_DIR}/{f}").num_rows for f in shard_files)
print(f"  TOTAL rows: {total_rows:,}")

# ── Schema check on first shard ──
print(f"\n{'='*60}")
print("SCHEMA INSPECTION (first shard)")
print(f"{'='*60}")
first_shard_path = f"{DISTILL_DIR}/{shard_files[0]}"
schema = pq.read_schema(first_shard_path)
print(f"\nColumns ({len(schema.names)}):")
for name in schema.names:
    print(f"  {name}: {schema.field(name).type}")

# ── Read first record and validate byte sizes ──
print(f"\n{'='*60}")
print("RECORD INTEGRITY CHECK (first 3 records)")
print(f"{'='*60}")
table = pq.read_table(first_shard_path).slice(0, 3)
df = table.to_pandas()

for i in range(min(3, len(df))):
    row = df.iloc[i]
    n = int(row["n_words"])
    print(f"\n  Record {i}: '{' '.join(list(row['words'])[:8])}{'...' if n > 8 else ''}'")
    print(f"    n_words={n}, verbs={row['per_verb_count']}, indices={row['per_verb_indices']}")

    # Check expected byte sizes
    checks = [
        ("pos_logits", n * N_POS * 2),
        ("ner_logits", n * N_NER * 2),
        ("dep_arc_scores", n * n * 2),
        ("cls_logits", N_CLS * 2),
        ("hidden_l12", n * H_t * 2),
        ("hidden_l18", n * H_t * 2),
        ("hidden_l24", n * H_t * 2),
        ("per_verb_srl_logits", row["per_verb_count"] * n * N_SRL * 2),
        ("per_verb_pred_hidden", row["per_verb_count"] * n * H_t * 2),
    ]
    all_ok = True
    for col, expected in checks:
        actual = len(row[col])
        ok = actual == expected
        if not ok:
            print(f"    ✗ {col}: {actual} bytes (expected {expected})")
            all_ok = False
    if all_ok:
        print(f"    ✓ All byte sizes match expected")

    # Decode one hidden state and check stats
    h24 = np.frombuffer(row["hidden_l24"], dtype=np.float16).reshape(n, H_t).astype(np.float32)
    print(f"    hidden_l24 stats: mean={h24.mean():.3f}, std={h24.std():.3f}, "
          f"min={h24.min():.2f}, max={h24.max():.2f}")

# ── Aggregate stats across all shards ──
print(f"\n{'='*60}")
print("AGGREGATE STATS (sample of 1000 across all shards)")
print(f"{'='*60}")
verb_counts = []
n_words_list = []
for f in shard_files[:2]:  # sample first 2 shards for speed
    table = pq.read_table(f"{DISTILL_DIR}/{f}", columns=["n_words", "per_verb_count"])
    df = table.to_pandas().sample(min(500, len(table)), random_state=42)
    verb_counts.extend(df["per_verb_count"].tolist())
    n_words_list.extend(df["n_words"].tolist())

print(f"  Sentence length: mean={np.mean(n_words_list):.1f}, "
      f"median={np.median(n_words_list):.0f}, max={max(n_words_list)}")
print(f"  Verbs per sentence: mean={np.mean(verb_counts):.2f}, "
      f"median={np.median(verb_counts):.0f}, max={max(verb_counts)}")
print(f"  Sentences with 0 verbs: {sum(1 for v in verb_counts if v == 0)} "
      f"({100*sum(1 for v in verb_counts if v == 0)/len(verb_counts):.1f}%)")
print(f"  Total predicted teacher SRL passes: ~{int(sum(verb_counts)/len(verb_counts) * total_rows):,} "
      f"(vs {total_rows:,} sentences)")

# ── Check metadata ──
if has_metadata:
    print(f"\n{'='*60}")
    print("METADATA")
    print(f"{'='*60}")
    with open(f"{DISTILL_DIR}/metadata.json") as f:
        meta = json.load(f)
    for k, v in meta.items():
        if isinstance(v, dict):
            print(f"  {k}:")
            for k2, v2 in v.items():
                print(f"    {k2}: {v2}")
        else:
            print(f"  {k}: {v}")
else:
    print(f"\n  ⚠  metadata.json missing — will need to write it manually")

print(f"\n{'='*60}")
print(f"✓ VERIFICATION COMPLETE")
print(f"{'='*60}")

DATA VERIFICATION

Directory: /content/drive/MyDrive/kniv-models/distillation_v2
  Shards found: 5
  metadata.json: yes

Shard breakdown:
  shard_000.parquet:  13.72 GB  (50,048 rows)
  shard_001.parquet:  13.71 GB  (50,048 rows)
  shard_002.parquet:  13.64 GB  (50,048 rows)
  shard_003.parquet:  13.83 GB  (50,048 rows)
  shard_004.parquet:   1.75 GB  ( 6,328 rows)
  ────────────────────────────────────────────────────────────
  TOTAL:  56.64 GB
  TOTAL rows: 206,520

SCHEMA INSPECTION (first shard)

Columns (17):
  words: list<element: string>
  n_words: int64
  pos_logits: binary
  ner_logits: binary
  dep_arc_scores: binary
  dep_label_top5: string
  cls_logits: binary
  pos_hard: list<element: string>
  ner_hard: list<element: string>
  hidden_l12: binary
  hidden_l18: binary
  hidden_l24: binary
  per_verb: string
  per_verb_count: int64
  per_verb_srl_logits: binary
  per_verb_pred_hidden: binary
  per_verb_indices: string

RECORD INTEGRITY CHECK (first 3 records)

  Record 0: 'H